# ALL Met Eyes

In [ ]:
from google.colab import userdata
PAT_XYZ = userdata.get("PAT_XYZ")

In [ ]:
!pip install mediapipe
!pip install ultralytics
!wget https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task
!wget https://raw.githubusercontent.com/acervos-digitais/met-faces-utils/refs/heads/main/utils.py
!wget https://raw.githubusercontent.com/acervos-digitais/met-faces-utils/refs/heads/main/utils_paintings.py
!git clone https://{PAT_XYZ}@github.com/acervos-digitais/met-faces-data.git data

In [ ]:
import json
import numpy as np
import requests

from os import makedirs, path
from PIL import Image as PImage
from time import sleep

from utils_paintings import PaintingsUtils

DATA_DIR = "./data"
IMG_DIR = f"{DATA_DIR}/image"
JSON_DIR = f"{DATA_DIR}/json"

JSON_OBJS_DIR = f"{JSON_DIR}/objects"
JSON_FACES_DIR = f"{JSON_DIR}/faces"
JSON_LANDMARKS_DIR = f"{JSON_DIR}/landmarks"

# makedirs(IMG_DIR, exist_ok=True)
makedirs(JSON_DIR, exist_ok=True)

mPU = PaintingsUtils(JSON_OBJS_DIR, JSON_FACES_DIR, JSON_LANDMARKS_DIR)

## The Met API

https://metmuseum.github.io/

https://github.com/metmuseum/openaccess

**Please limit request rate to 80 requests per second.**

### Painting Objects

- $14\text{,}982$ paintings available in API
- $14\text{,}200$ have images

In [ ]:
obj_ids = PaintingsUtils.get_object_ids()
len(obj_ids)

### Get Object Metadata, Faces and Landmarks

In [ ]:
def save_eye_pairs(obj_data, img):
  obj_data = json.loads(json.dumps(obj_data))
  oid = obj_data["objectID"]
  iw,ih = img.size
  # TODO: for loop
  #   TODO: check file
  #   TODO: extract and save

In [ ]:
for cnt,oid in enumerate(obj_ids[:1024]):
  if cnt % 64 == 0:
    print(f"{cnt} / {len(obj_ids)}")

  json_landmark_path = f"{JSON_LANDMARKS_DIR}/{oid}.json"
  if path.isfile(json_landmark_path):
    continue

  obj_data = mPU.get_obj_data(oid)

  if obj_data is None:
    continue

  img_url = obj_data["primaryImage"]
  img_response = requests.get(img_url, stream=True)
  img = PImage.open(img_response.raw)

  face_data = mPU.get_face_data(obj_data, img)

  landmark_data = mPU.get_landmark_data(face_data, img)

  # TODO: crop eyes from img and save avif
  save_eye_pairs(landmark_data, img)

  sleep(0.25)

### Export Combined Object Metadata

In [ ]:
export_combined_jsons(JSON_OBJS_DIR, JSON_DIR, "objects")
export_combined_jsons(JSON_FACES_DIR, JSON_DIR, "faces")
export_combined_jsons(JSON_LANDMARK_DIR, JSON_DIR, "landmarks")